### Data Download for the Two Candidate Problems (from website, not local)

In [1]:

import pandas as pd
import requests

def load_socrata(url, columns, where, batch_size=10000):
    """Load filtered Socrata data directly into a pandas DataFrame."""
    frames = []
    offset = 0

    while True:
        params = {
            "$select": ",".join(columns),
            "$where": where,
            "$order": columns[0] + " ASC",
            "$limit": batch_size,
            "$offset": offset
        }

        response = requests.get(url, params=params, timeout=120)
        response.raise_for_status()
        batch = response.json()

        if not batch:
            break

        frames.append(pd.DataFrame(batch))
        offset += len(batch)
        print(f"Loaded {offset:,} rows...", end="\r")

        if len(batch) < batch_size:
            break

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)


1. Chicago Crashes Data

In [4]:

chicago_url = "https://data.cityofchicago.org/resource/85ca-t3if.json"


chicago_columns = [
    "crash_record_id",
    "crash_date",
    "injuries_total",
    "posted_speed_limit",
    "weather_condition",
    "lighting_condition",
    "roadway_surface_cond",
    "traffic_control_device",
    "device_condition",
    "trafficway_type",
    "alignment",
    "road_defect",
    "intersection_related_i",
    "lane_cnt",
    "street_direction",
    "first_crash_type"
]


chicago_where = (
    "crash_date >= '2023-01-01T00:00:00' "
    "AND crash_date < '2026-01-01T00:00:00' "
    "AND injuries_total IS NOT NULL"
)

chicago = load_socrata(
    chicago_url,
    chicago_columns,
    chicago_where
)

chicago["crash_date"] = pd.to_datetime(chicago["crash_date"])
chicago["injuries_total"] = pd.to_numeric(chicago["injuries_total"])

chicago["hour"] = chicago["crash_date"].dt.hour
chicago["day_of_week"] = chicago["crash_date"].dt.dayofweek
chicago["month"] = chicago["crash_date"].dt.month

chicago["multiple_injuries"] = (
    chicago["injuries_total"] >= 2
).astype(int)

print("Chicago observations:", len(chicago))
print("Chicago columns:", chicago.shape[1])
print(chicago["multiple_injuries"].value_counts())
chicago.head()


Chicago observations: 331203
Chicago columns: 20
multiple_injuries
0    317496
1     13707
Name: count, dtype: int64


,crash_record_id,crash_date,injuries_total,posted_speed_limit,weather_condition,lighting_condition,roadway_surface_cond,traffic_control_device,device_condition,trafficway_type,alignment,road_defect,street_direction,first_crash_type,intersection_related_i,lane_cnt,hour,day_of_week,month,multiple_injuries
0,00005696946846c8b8a1d378dba4e2a5ed84a9b2876fe0...,2024-02-02 09:48:00,0.0,30,CLEAR,DAYLIGHT,DRY,NO CONTROLS,NO CONTROLS,OTHER,STRAIGHT AND LEVEL,NO DEFECTS,S,FIXED OBJECT,NaN,NaN,9,4,2,0
1,00007d9c4ff8c756eedfd371c6aae35ede63de4400660f...,2025-09-11 16:44:00,0.0,30,CLEAR,DAYLIGHT,DRY,NO CONTROLS,NO CONTROLS,NOT DIVIDED,STRAIGHT AND LEVEL,NO DEFECTS,W,SIDESWIPE SAME DIRECTION,NaN,NaN,16,3,9,0
2,0000b8e88de2441e6d530c9ee6a75c216a64ad07caa905...,2025-01-30 06:30:00,0.0,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,NO CONTROLS,NO CONTROLS,DIVIDED - W/MEDIAN (NOT RAISED),STRAIGHT AND LEVEL,NO DEFECTS,S,ANGLE,NaN,NaN,6,3,1,0
3,0000f068043805e104071e24086b0871390fd09737affb...,2024-10-21 15:12:00,0.0,30,CLEAR,DAYLIGHT,DRY,NO CONTROLS,NO CONTROLS,NOT DIVIDED,STRAIGHT AND LEVEL,UNKNOWN,W,SIDESWIPE SAME DIRECTION,NaN,NaN,15,0,10,0
4,00011726c1086305fb18fed9fd00703c02fd89917e47a1...,2024-08-04 14:00:00,0.0,20,CLEAR,DAYLIGHT,DRY,NO CONTROLS,NO CONTROLS,ONE-WAY,STRAIGHT AND LEVEL,NO DEFECTS,N,PARKED MOTOR VEHICLE,NaN,NaN,14,6,8,0


2. EMS Response Times in the Bronx

In [5]:

ems_url = "https://data.cityofnewyork.us/resource/76xm-jjuj.json"

ems_columns = [
    "incident_id",
    "incident_datetime",
    "initial_call_type",
    "initial_severity_level_code",
    "incident_dispatch_area",
    "zipcode",
    "incident_response_seconds_qy",
    "valid_incident_rspns_time_indc"
]

ems_where = (
    "incident_datetime >= '2023-01-01T00:00:00' "
    "AND incident_datetime < '2026-01-01T00:00:00' "
    "AND incident_dispatch_area IN ('B1','B2','B3','B4','B5') "
    "AND initial_severity_level_code IN ('1','2','3')"
)

ems = load_socrata(
    ems_url,
    ems_columns,
    ems_where
)

ems["incident_datetime"] = pd.to_datetime(ems["incident_datetime"])

ems["incident_response_seconds_qy"] = pd.to_numeric(
    ems["incident_response_seconds_qy"],
    errors="coerce"
)

ems = ems[
    (ems["valid_incident_rspns_time_indc"] == "Y") &
    (ems["incident_response_seconds_qy"] > 0)
].copy()

ems["hour"] = ems["incident_datetime"].dt.hour
ems["day_of_week"] = ems["incident_datetime"].dt.dayofweek
ems["month"] = ems["incident_datetime"].dt.month
ems["weekend"] = (ems["day_of_week"] >= 5).astype(int)

ems["rush_hour"] = (
    (ems["day_of_week"] < 5) &
    (
        ems["hour"].between(7, 9) |
        ems["hour"].between(16, 18)
    )
).astype(int)

print("EMS observations:", len(ems))
print("EMS columns:", ems.shape[1])
print(ems["incident_dispatch_area"].value_counts())
ems.head()


EMS observations: 417956
EMS columns: 13
incident_dispatch_area
B2    122567
B1    100274
B3     81532
B5     60356
B4     53227
Name: count, dtype: int64


,incident_id,incident_datetime,initial_call_type,initial_severity_level_code,incident_dispatch_area,zipcode,incident_response_seconds_qy,valid_incident_rspns_time_indc,hour,day_of_week,month,weekend,rush_hour
0,76517036,2023-01-01 00:00:30,DIFFFC,2,B5,10463,479.0,Y,0,6,1,1,0
1,76517056,2023-01-01 00:06:51,ALTMEN,3,B3,10461,675.0,Y,0,6,1,1,0
2,76517073,2023-01-01 00:12:47,PD13C,3,B5,10458,302.0,Y,0,6,1,1,0
3,76517075,2023-01-01 00:14:01,CARDBR,2,B5,10467,674.0,Y,0,6,1,1,0
4,76517137,2023-01-01 00:27:00,UNC,2,B1,10460,695.0,Y,0,6,1,1,0
